# EDA - Proyecto Inmuebles

Notebook para analisis exploratorio sobre el dataset final para BI.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 200)
sns.set_theme(style='whitegrid')

In [2]:
DATA_PATH = '../pre-procesamiento/processed_powerbi_es.csv'
df = pd.read_csv(DATA_PATH, sep=';', decimal=',', low_memory=False)

print('Shape:', df.shape)
print('Columnas:', len(df.columns))
display(df.head())

Shape: (144961, 39)
Columnas: 39


,country,location,latitude,longitude,un_continent,un_statistical,price_in_Euro,entry_price_eur,building_construction_year,apartament_m2,building_total_floors,apartment_floor,apartment_rooms,apartment_bedrooms,apartment_bathrooms,apartment_total_area,apartment_living_area,riesgo_ocupacion,leyes_de_ocupacion_comentario,leyes_de_ocupacion_url,pib_millones_ppa,gini,gini_year,population_2018,population_2019,population_change,crime_index,safety_index,unemployment_last,unemployment_previous,unemployment_reference,country_code,country_iso_code,price_in_Euro_calculo,apartment_total_area_calculo,apartment_living_area_calculo,image,url,country_norm_merge_key
0,Armenia,Yerevan,40.177711,44.512623,Asia,Western Asia,279.000 €,83.700 €,2022,mansion,3,5,3,2,1,280 m²,86 m²,medio,Propiedad privada protegida; ocupacion persegu...,https://www.arlis.am/,42058,34.4,2018.0,2951745,2957731,0.2,22.69,77.31,17.5,19.8,20-Jun,AM,ARM,279000.00,280.0,86.0,https://realting.com/uploads/bigSlider/493/0af...,https://realting.com/property-for-sale/armenia...,armenia
1,Armenia,"Dilijan, Tavush Province, Ijevan Region",40.069099,45.038189,Asia,Western Asia,279.000 €,83.700 €,2022,mansion,1,1,3,2,1,6 m²,86 m²,medio,Propiedad privada protegida; ocupacion persegu...,https://www.arlis.am/,42058,34.4,2018.0,2951745,2957731,0.2,22.69,77.31,17.5,19.8,20-Jun,AM,ARM,279000.00,6.0,86.0,https://realting.com/uploads/bigSlider/e12/552...,https://realting.com/property-for-sale/armenia...,armenia
2,Armenia,Yerevan,40.177711,44.512623,Asia,Western Asia,"160.158,09 €","48.047,43 €",2022,apartment,2,1,2,1,1,80 m²,86 m²,medio,Propiedad privada protegida; ocupacion persegu...,https://www.arlis.am/,42058,34.4,2018.0,2951745,2957731,0.2,22.69,77.31,17.5,19.8,20-Jun,AM,ARM,160158.09,80.0,86.0,https://realting.com/uploads/bigSlider/048/1d1...,https://realting.com/property-for-sale/armenia...,armenia
3,Armenia,Yerevan,40.177711,44.512623,Asia,Western Asia,167.400 €,50.220 €,2022,apartment,14,14,3,2,1,96 m²,86 m²,medio,Propiedad privada protegida; ocupacion persegu...,https://www.arlis.am/,42058,34.4,2018.0,2951745,2957731,0.2,22.69,77.31,17.5,19.8,20-Jun,AM,ARM,167400.00,96.0,86.0,https://realting.com/uploads/bigSlider/712/1aa...,https://realting.com/property-for-sale/armenia...,armenia
4,Armenia,Yerevan,40.177711,44.512623,Asia,Western Asia,344.100 €,103.230 €,2022,apartment,9,8,5,4,1,124 m²,86 m²,medio,Propiedad privada protegida; ocupacion persegu...,https://www.arlis.am/,42058,34.4,2018.0,2951745,2957731,0.2,22.69,77.31,17.5,19.8,20-Jun,AM,ARM,344100.00,124.0,86.0,https://realting.com/uploads/bigSlider/fb0/723...,https://realting.com/property-for-sale/armenia...,armenia


In [4]:
# KPI 1: Total de inmuebles
total_inmuebles = int(len(df))
print(f'Total de inmuebles: {total_inmuebles:,}')

Total de inmuebles: 144,961


In [5]:
# KPI 2: Total de inmuebles por nivel de riesgo
riesgo_col = 'riesgo_ocupacion'

if riesgo_col in df.columns:
    riesgo_counts = (
        df[riesgo_col]
        .fillna('sin_dato')
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({'': 'sin_dato'})
        .value_counts()
    )

    total_alto = int(riesgo_counts.get('alto', 0))
    total_medio = int(riesgo_counts.get('medio', 0))
    total_bajo = int(riesgo_counts.get('bajo', 0))

    print(f'Total riesgo alto: {total_alto:,}')
    print(f'Total riesgo medio: {total_medio:,}')
    print(f'Total riesgo bajo: {total_bajo:,}')

    display(riesgo_counts.rename_axis('riesgo_ocupacion').reset_index(name='total'))
else:
    print("No existe la columna 'riesgo_ocupacion' en el dataset.")

Total riesgo alto: 36,093
Total riesgo medio: 51,725
Total riesgo bajo: 57,143


,riesgo_ocupacion,total
0,bajo,57143
1,medio,51725
2,alto,36093


In [6]:
# KPI 3: Lista de paises con porcentaje de criminalidad (crime_index)
if {'country', 'crime_index'}.issubset(df.columns):
    crime_country = (
        df.assign(crime_index_num=pd.to_numeric(df['crime_index'], errors='coerce'))
          .dropna(subset=['country', 'crime_index_num'])
          .groupby('country', as_index=False)
          .agg(
              porcentaje_criminalidad=('crime_index_num', 'mean'),
              total_inmuebles=('country', 'size')
          )
          .sort_values('porcentaje_criminalidad', ascending=False)
          .reset_index(drop=True)
    )

    crime_country['porcentaje_criminalidad'] = crime_country['porcentaje_criminalidad'].round(2)

    print(f'Paises evaluados: {len(crime_country)}')
    display(crime_country)
else:
    print("No existen columnas requeridas: 'country' y/o 'crime_index'.")

Paises evaluados: 27


,country,porcentaje_criminalidad,total_inmuebles
0,United States,47.70,414
1,Indonesia,46.06,102
2,Italy,44.24,3002
3,Australia,41.67,1
4,Montenegro,41.56,10308
5,Greece,41.30,12341
6,Russia,40.60,18481
7,Thailand,40.01,1968
8,Turkey,39.50,25648
9,Northern Cyprus,39.50,745


In [7]:
# KPI 4: Paises mas baratos con menor riesgo de ocupacion
required_cols = {'country', 'riesgo_ocupacion', 'price_in_Euro_calculo'}

if required_cols.issubset(df.columns):
    work = df.copy()
    work['precio_eur_num'] = pd.to_numeric(work['price_in_Euro_calculo'], errors='coerce')
    work['riesgo_norm'] = (
        work['riesgo_ocupacion']
        .fillna('sin_dato')
        .astype(str)
        .str.strip()
        .str.lower()
    )

    agg = (
        work.dropna(subset=['country', 'precio_eur_num'])
            .groupby('country', as_index=False)
            .agg(
                total_inmuebles=('country', 'size'),
                precio_medio_eur=('precio_eur_num', 'mean'),
                pct_riesgo_bajo=('riesgo_norm', lambda s: (s.eq('bajo').mean() * 100)),
                pct_riesgo_medio=('riesgo_norm', lambda s: (s.eq('medio').mean() * 100)),
                pct_riesgo_alto=('riesgo_norm', lambda s: (s.eq('alto').mean() * 100)),
            )
    )

    # Candidatos con riesgo bajo dominante
    candidatos = agg[agg['pct_riesgo_bajo'] >= 50].copy()

    ranking_barato_seguro = (
        candidatos.sort_values(['precio_medio_eur', 'pct_riesgo_alto', 'pct_riesgo_bajo'], ascending=[True, True, False])
                  .reset_index(drop=True)
    )

    for c in ['precio_medio_eur', 'pct_riesgo_bajo', 'pct_riesgo_medio', 'pct_riesgo_alto']:
        ranking_barato_seguro[c] = ranking_barato_seguro[c].round(2)

    top10 = ranking_barato_seguro.head(10)

    print('Top 10 paises mas baratos con menor riesgo de ocupacion (riesgo bajo >= 50%):')
    print(top10[['country', 'precio_medio_eur', 'pct_riesgo_bajo', 'pct_riesgo_alto', 'total_inmuebles']].to_string(index=False))
    display(top10)
else:
    print('Faltan columnas requeridas para este analisis.')

Top 10 paises mas baratos con menor riesgo de ocupacion (riesgo bajo >= 50%):
        country  precio_medio_eur  pct_riesgo_bajo  pct_riesgo_alto  total_inmuebles
        Belarus          66827.11            100.0              0.0            13742
        Georgia          87222.28            100.0              0.0             3291
     Uzbekistan         104856.49            100.0              0.0             1950
      Lithuania         163625.71            100.0              0.0             2567
Northern Cyprus         174580.62            100.0              0.0              745
         Cyprus         181898.17            100.0              0.0               79
        Finland         207512.72            100.0              0.0              816
 Czech Republic         255896.28            100.0              0.0             1371
       Thailand         261075.72            100.0              0.0             1968
      Indonesia         319930.24            100.0              0.0     

,country,total_inmuebles,precio_medio_eur,pct_riesgo_bajo,pct_riesgo_medio,pct_riesgo_alto
0,Belarus,13742,66827.11,100.0,0.0,0.0
1,Georgia,3291,87222.28,100.0,0.0,0.0
2,Uzbekistan,1950,104856.49,100.0,0.0,0.0
3,Lithuania,2567,163625.71,100.0,0.0,0.0
4,Northern Cyprus,745,174580.62,100.0,0.0,0.0
5,Cyprus,79,181898.17,100.0,0.0,0.0
6,Finland,816,207512.72,100.0,0.0,0.0
7,Czech Republic,1371,255896.28,100.0,0.0,0.0
8,Thailand,1968,261075.72,100.0,0.0,0.0
9,Indonesia,102,319930.24,100.0,0.0,0.0


In [3]:
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'nulls': df.isna().sum(),
    'null_pct': (df.isna().mean() * 100).round(2),
    'n_unique': df.nunique(dropna=True)
}).sort_values(['null_pct', 'n_unique'], ascending=[False, False])

display(summary.head(25))

,dtype,nulls,null_pct,n_unique
location,str,4474,3.09,7173
gini_year,float64,875,0.60,6
country,str,130,0.09,27
country_norm_merge_key,str,130,0.09,27
url,str,0,0.00,144961
image,str,0,0.00,111402
entry_price_eur,str,0,0.00,53670
price_in_Euro,str,0,0.00,49327
price_in_Euro_calculo,float64,0,0.00,49327
apartment_total_area,str,0,0.00,899
